
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [4]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [5]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [12]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    all_summaries = []
    for i, batch in enumerate(batch_generator(texts, batch_size)):
        # Prefix inputs with "summarize: "
        prefixed_batch = ["summarize: " + text for text in batch]

        inputs = tokenizer(prefixed_batch, return_tensors="pt", padding=True, truncation=True).to(device)

        # Generate summaries
        with torch.no_grad():
            outputs = model.generate(inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=max_new_tokens)

        # Decode with skip_special_tokens=True
        summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_summaries.extend(summaries)

        # Clear CUDA cache between batches
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,reference_summary,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [13]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

# train_summaries_t5 is now available, so we can directly compute accuracy
acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
print(f"Exact-match accuracy: {acc:.4f}")

Exact-match accuracy: 0.0000



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [14]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    # Join sentences with newlines, as recommended for ROUGE-L
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # Normalize predictions and references
    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]

    # Compute ROUGE scores
    results = rouge.compute(
        predictions=normalized_preds,
        references=normalized_refs,
        use_stemmer=True  # Use stemming for better recall
    )
    return results

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}


In [15]:
# Part VI: Understanding ROUGE scores

print("### 1. Exact match vs. Empty prediction ###")
preds_exact = ["The quick brown fox jumps over the lazy dog."]
refs_exact = ["The quick brown fox jumps over the lazy dog."]
preds_empty = [""]
refs_empty = ["The quick brown fox jumps over the lazy dog."]

print("Exact match ROUGE:")
print(compute_rouge_score(preds_exact, refs_exact))
print("Empty prediction ROUGE:")
print(compute_rouge_score(preds_empty, refs_empty))

print("\n### 2. Effect of Stemming ###")
preds_stem = ["He is running fast."]
refs_stem = ["He runs fast."]

# We will use the 'use_stemmer=True' default in compute_rouge_score
print("Running (pred) vs. Runs (ref) with stemming:")
print(compute_rouge_score(preds_stem, refs_stem))

# To demonstrate without stemming, we'd need to modify the function or call directly
# For simplicity, we assume compute_rouge_score handles stemming internally as implemented

print("\n### 3. N-gram Overlap (ROUGE-1 vs. ROUGE-2) ###")
preds_ngram_1 = ["The cat sat."]
refs_ngram_1 = ["The dog sat."] # 1-gram overlap (The, sat)

preds_ngram_2 = ["The brown cat sat."]
refs_ngram_2 = ["The lazy cat sat."] # 1-gram overlap (The, cat, sat), 2-gram overlap (cat sat)

print("Partial overlap (1-gram):")
print(compute_rouge_score(preds_ngram_1, refs_ngram_1))
print("More overlap (1-gram, 2-gram):")
print(compute_rouge_score(preds_ngram_2, refs_ngram_2))

print("\n### 4. Symmetry (Swapping preds/refs) ###")
preds_symm = ["A B C D E"]
refs_symm = ["A B C"]

print("Original (preds, refs):")
original_score = compute_rouge_score(preds_symm, refs_symm)
print(original_score)

print("Swapped (refs, preds):")
swapped_score = compute_rouge_score(refs_symm, preds_symm)
print(swapped_score)

print("\nInterpretation:")
print("- Exact Match: ROUGE scores are perfect (1.0) when predictions and references are identical.")
print("- Empty Prediction: ROUGE scores are 0.0 because there is no overlap with the reference.")
print("- Stemming: Using a stemmer ('use_stemmer=True') helps match words with different inflections (e.g., 'running' and 'runs'), leading to higher ROUGE scores than without it. For the 'running' vs 'runs' example, the scores will reflect the overlap after stemming both to 'run'.")
print("- N-gram Overlap: ROUGE-1 measures unigram (single word) overlap, ROUGE-2 measures bigram (two-word sequence) overlap. Higher overlap leads to higher scores. ROUGE-2 is stricter as it requires sequential word matches.")
print("- Symmetry: ROUGE-N and ROUGE-L are recall-oriented, meaning they measure how much of the reference is captured by the prediction. Therefore, swapping predictions and references will change the scores significantly, as the 'reference' length changes. For instance, if the prediction is much longer than the reference but contains all its key info, recall will be high, but if the reference is much longer than the prediction, recall will be low.")

### 1. Exact match vs. Empty prediction ###
Exact match ROUGE:
{'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}
Empty prediction ROUGE:
{'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}

### 2. Effect of Stemming ###
Running (pred) vs. Runs (ref) with stemming:
{'rouge1': np.float64(0.8571428571428571), 'rouge2': np.float64(0.4), 'rougeL': np.float64(0.8571428571428571), 'rougeLsum': np.float64(0.8571428571428571)}

### 3. N-gram Overlap (ROUGE-1 vs. ROUGE-2) ###
Partial overlap (1-gram):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}
More overlap (1-gram, 2-gram):
{'rouge1': np.float64(0.75), 'rouge2': np.float64(0.3333333333333333), 'rougeL': np.float64(0.75), 'rougeLsum': np.float64(0.75)}

### 4. Symmetry (Swapping preds/refs) ###
Original (preds, ref


### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import torch, gc

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # GPT-2 does not have a padding token by default.
    # It's common practice to set the pad token to the eos token for generation.
    if tokenizer.pad_token is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.eos_token_id

    # For decoder-only models like GPT-2, padding on the left is often preferred for generation.
    tokenizer.padding_side = "left"

    all_summaries = []
    for batch_texts in batch_generator(texts, batch_size):
        prefixed_batch = [text + "\nTL;DR:" for text in batch_texts]

        # Adjust max_length for tokenizer to ensure input + generated tokens don't exceed model's max_position_embeddings
        # GPT-2 has a max_position_embeddings of 1024. If max_new_tokens is 32, max_length for input should be 1024 - 32 = 992.
        max_input_length = model.config.max_position_embeddings - max_new_tokens
        inputs = tokenizer(
            prefixed_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # For reproducible results
                num_beams=1,      # Greedy decoding for simplicity
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id # Explicitly pass here
            )

        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        summaries = []
        for decoded_output in decoded_outputs:
            # Find the start of the generated summary (after original text + prefix)
            # Use a more robust way to extract the summary after "TL;DR:"
            tldr_idx = decoded_output.rfind("TL;DR:")
            if tldr_idx != -1:
                summary_text = decoded_output[tldr_idx + len("TL;DR:"):].strip()
                # Take only the first line after TL;DR to avoid extraneous text
                summary_text = summary_text.split('\n')[0].strip()
            else:
                summary_text = decoded_output.strip() # Fallback
            summaries.append(summary_text)

        all_summaries.extend(summaries)

        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    rouge_scores = []
    # Ensure that `rouge` object is loaded globally or passed if needed
    if 'rouge' not in globals():
        global rouge
        import evaluate
        rouge = evaluate.load('rouge')

    for index, row in df.iterrows():
        pred = normalize_text(row[pred_col])
        ref = normalize_text(row[ref_col])

        # Compute ROUGE for a single prediction-reference pair
        scores = rouge.compute(predictions=[pred], references=[ref], use_stemmer=True)
        rouge_scores.append({
            f'{pred_col}_rouge1': scores['rouge1'],
            f'{pred_col}_rouge2': scores['rouge2'],
            f'{pred_col}_rougeL': scores['rougeL'],
            f'{pred_col}_rougeLsum': scores['rougeLsum']
        })
    return pd.DataFrame(rouge_scores)

RUN_COMPARE = True # Set to True to enable model comparison
if RUN_COMPARE:
    print("Generating summaries with t5-small...")
    train_summaries_t5_small = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2, max_new_tokens=32)
    train_df['t5_small_summary'] = train_summaries_t5_small

    print("Generating summaries with t5-base...")
    train_summaries_t5_base = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base', batch_size=2, max_new_tokens=32)
    train_df['t5_base_summary'] = train_summaries_t5_base

    print("Generating summaries with gpt2...")
    train_summaries_gpt2 = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2, max_new_tokens=32)
    train_df['gpt2_summary'] = train_summaries_gpt2

    # Compute ROUGE scores per row for each model
    print("Computing ROUGE scores for t5-small...")
    t5_small_rouge_df = compute_rouge_per_row(train_df, 't5_small_summary')
    train_df = pd.concat([train_df, t5_small_rouge_df], axis=1)

    print("Computing ROUGE scores for t5-base...")
    t5_base_rouge_df = compute_rouge_per_row(train_df, 't5_base_summary')
    train_df = pd.concat([train_df, t5_base_rouge_df], axis=1)

    print("Computing ROUGE scores for gpt2...")
    gpt2_rouge_df = compute_rouge_per_row(train_df, 'gpt2_summary')
    train_df = pd.concat([train_df, gpt2_rouge_df], axis=1)

    display(train_df.head())
else:
    print("Model comparison skipped. Set RUN_COMPARE=True to execute.")

Generating summaries with t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating summaries with t5-base...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Generating summaries with gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Computing ROUGE scores for t5-small...
Computing ROUGE scores for t5-base...
Computing ROUGE scores for gpt2...


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_small_summary_rougeLsum,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,t5_base_summary_rougeLsum,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL,gpt2_summary_rougeLsum
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,The race was a bit of a mess for Lewis Hamilto...,0.222222,0.131148,0.222222,0.222222,0.507937,0.262295,0.444444,0.476190,0.179104,0.092308,0.179104,0.179104
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: china suspends exports of the Aqua Dot...,China suspended exports of Aqua Dots toys cont...,0.147059,0.000000,0.117647,0.117647,0.212121,0.031250,0.151515,0.181818,0.149254,0.030769,0.089552,0.119403
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a company that is not only a leade...,0.424242,0.156250,0.333333,0.393939,0.242424,0.031250,0.181818,0.242424,0.281250,0.032258,0.187500,0.281250
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: president pervez muharraf says his action...,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",0.237288,0.035088,0.203390,0.169492,0.369231,0.253968,0.307692,0.369231,0.096357,0.054181,0.072855,0.094007
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Linds...,0.413793,0.142857,0.275862,0.413793,0.533333,0.241379,0.333333,0.533333,0.542373,0.210526,0.305085,0.440678



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [20]:
import pandas as pd

def compare_models(rouge_dataframes: dict, metrics: list = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']):
    results = {} # Store model_name: average_scores
    for model_name, df in rouge_dataframes.items():
        avg_scores = {
            f'avg_{metric}': df[metric].mean() # Corrected line: Access metric directly
            for metric in metrics
        }
        results[model_name] = avg_scores

    # Convert results to a DataFrame for better display
    return pd.DataFrame.from_dict(results, orient='index')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list, ref_col: str = 'prompt_title', num_samples: int = 5):
    # Select a random sample of rows
    sample_df = df.sample(n=min(num_samples, len(df)), random_state=42)

    # Prepare columns for display
    display_cols = [ref_col] + pred_cols

    return sample_df[display_cols].reset_index(drop=True)

# --- Execution of Part VIII ---
# Assume train_df now contains 't5_small_summary', 't5_base_summary', 'gpt2_summary'
# and their respective ROUGE scores as columns.

# Extract ROUGE score DataFrames (if not already separate)
# For simplicity, we'll recreate them from the existing train_df or use the ones from kernel state.
# In a real scenario, you'd manage these separately or ensure column names are consistent.

# Assuming `train_df` has been updated with all model summaries and their ROUGE scores
# from the previous cell's execution.

# Prepare the data for compare_models function
rouge_dfs_for_comparison = {
    't5_small': train_df[['t5_small_summary_rouge1', 't5_small_summary_rouge2', 't5_small_summary_rougeL', 't5_small_summary_rougeLsum']].rename(columns=lambda x: x.replace('t5_small_summary_', '')),
    't5_base': train_df[['t5_base_summary_rouge1', 't5_base_summary_rouge2', 't5_base_summary_rougeL', 't5_base_summary_rougeLsum']].rename(columns=lambda x: x.replace('t5_base_summary_', '')),
    'gpt2': train_df[['gpt2_summary_rouge1', 'gpt2_summary_rouge2', 'gpt2_summary_rougeL', 'gpt2_summary_rougeLsum']].rename(columns=lambda x: x.replace('gpt2_summary_', ''))
}

# Aggregate and display average ROUGE scores
print("\n### Aggregated ROUGE Scores Across Models ###")
aggregated_rouge_scores = compare_models(rouge_dfs_for_comparison)
display(aggregated_rouge_scores)

# Display side-by-side summary comparisons
print("\n### Side-by-Side Summary Comparisons (Sample) ###")
summary_comparison_df = compare_models_summaries(train_df, ['t5_small_summary', 't5_base_summary', 'gpt2_summary'])
display(summary_comparison_df)


### Aggregated ROUGE Scores Across Models ###


,avg_rouge1,avg_rouge2,avg_rougeL,avg_rougeLsum
t5_small,0.278179,0.098507,0.210632,0.257997
t5_base,0.303621,0.116064,0.224504,0.284265
gpt2,0.180748,0.050708,0.133987,0.168448



### Side-by-Side Summary Comparisons (Sample) ###


,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,Cpl. Trent D. Thomas found guilty this week of...,a marine convicted of his role in the death of...,"marine convicted of kidnapping, conspiracy to ...","The Marine Corps is now a ""bad-conduct dischar..."
1,"India elects first female president, official ...","a woman is murdered, raped or abused every thr...","india elects its first female president, offic...",Pratibha Patil is a woman who has been elected...
2,Taliban militants kill Australian commando in ...,four australia troops have died in the conflic...,"one australian soldier, three civilians and Ta...",The Australian soldier was killed in the fight...
3,Savers at leading UK mortgage bank lined up to...,fears of customers pulling cash from a leading...,customers at leading mortgage bank line up for...,The Bank of England is now facing a crisis of ...
4,"Two employees bought, sold weapons on their ow...",new: prosecutors investigating allegations tha...,federal prosecutors investigating allegations ...,Blackwater USA is a private security firm that...



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
